# N-BEATS Experiment — Clean Version

This notebook keeps the experiment flow explicit:

1. Setup and data loading
2. Common preprocessing from `preprocessing.py`
3. N-BEATS-specific preprocessing
4. Baseline
5. Train-tail check for overfit/underfit signal
6. Validation training + evaluation
7. MLflow/DagsHub logging
8. Save model only if it improves the best `valid_wmae`

**Important:** This N-BEATS version is a global univariate model. It uses historical sales only: `unique_id`, `ds`, `y`. Other preprocessing features are created but not passed to N-BEATS v1.


## 1. Setup repo, paths, and packages

In [12]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.git"
REPO_DIR = Path("/content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "src"))

print("Working directory:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting


In [13]:
!pip install -q mlflow neuralforecast utilsforecast dagshub

## 2. Download Kaggle data if needed

In [14]:
DATA_DIR = REPO_DIR / "data" / "raw"
PROCESSED_DIR = REPO_DIR / "data" / "processed"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    DATA_DIR / "train.csv.zip",
    DATA_DIR / "test.csv.zip",
    DATA_DIR / "features.csv.zip",
    DATA_DIR / "stores.csv",
]

if not all(p.exists() for p in required_files):
    print("Raw Kaggle files not found. Upload kaggle.json when prompted.")
    from google.colab import files
    files.upload()

    Path.home().joinpath(".kaggle").mkdir(exist_ok=True)
    subprocess.run(["cp", "kaggle.json", str(Path.home() / ".kaggle" / "kaggle.json")], check=True)
    subprocess.run(["chmod", "600", str(Path.home() / ".kaggle" / "kaggle.json")], check=True)

    subprocess.run([
        "kaggle", "competitions", "download",
        "-c", "walmart-recruiting-store-sales-forecasting",
        "-p", str(DATA_DIR)
    ], check=True)

    subprocess.run([
        "unzip", "-o",
        str(DATA_DIR / "walmart-recruiting-store-sales-forecasting.zip"),
        "-d", str(DATA_DIR)
    ], check=True)
else:
    print("Raw Kaggle files already exist. Skipping download.")

Raw Kaggle files already exist. Skipping download.


## 3. Run or load common preprocessing

In [15]:
from preprocessing import run_pipeline, load_processed, weighted_mae

processed_files = [
    PROCESSED_DIR / "train_part.parquet",
    PROCESSED_DIR / "valid_part.parquet",
    PROCESSED_DIR / "train_full.parquet",
    PROCESSED_DIR / "test_full.parquet",
]

if all(p.exists() for p in processed_files):
    print("Loading existing processed parquet files...")
    train_part, valid_part, train_full, test_full = load_processed(PROCESSED_DIR)
else:
    print("Running common preprocessing...")
    train_part, valid_part, train_full, test_full = run_pipeline(
        data_dir=DATA_DIR,
        out_dir=PROCESSED_DIR,
        months_valid=3,
        save=True,
    )

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)
print("train_full:", train_full.shape)
print("test_full:", test_full.shape)

Loading existing processed parquet files...
train_part: (429699, 33)
valid_part: (46634, 33)
train_full: (476333, 33)
test_full: (115064, 29)


## 4. Experiment config

In [56]:
MODEL_NAME = "NBEATS"

CONFIG = {
    "model": MODEL_NAME,
    "model_type": "global_univariate",
    "target_transform": "none",   # options: "none", "log1p"
    "input_size": 104,
    "horizon": valid_part["Date"].nunique(),
    "loss": "MAE",
    "learning_rate": 5e-4,
    "batch_size": 32,
    "max_steps": 1500,
    "validation_split": "last_3_months",
    "freq": "W-FRI",
    "random_seed": 42,
    "scaler_type": None,          # optionally test "robust" in a separate run
    "missing_target_strategy": "linear_interpolation",
}

RUN_NAME = (
    f"NBEATS_{CONFIG['target_transform']}_target"
    f"_input{CONFIG['input_size']}"
    f"_h{CONFIG['horizon']}"
    f"_steps{CONFIG['max_steps']}"
)

CONFIG, RUN_NAME

({'model': 'NBEATS',
  'model_type': 'global_univariate',
  'target_transform': 'none',
  'input_size': 104,
  'horizon': 14,
  'loss': 'MAE',
  'learning_rate': 0.0005,
  'batch_size': 32,
  'max_steps': 2000,
  'validation_split': 'last_3_months',
  'freq': 'W-FRI',
  'random_seed': 42,
  'scaler_type': None,
  'missing_target_strategy': 'linear_interpolation'},
 'NBEATS_none_target_input104_h14_steps2000')

## 5. N-BEATS-specific preprocessing helpers

In [34]:
import numpy as np
import pandas as pd
from typing import Dict, Tuple

def fix_isholiday_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Make IsHoliday stable after merges that may create IsHoliday_x / IsHoliday_y."""
    df = df.copy()

    if "IsHoliday" not in df.columns:
        if "IsHoliday_x" in df.columns:
            df["IsHoliday"] = df["IsHoliday_x"]
        elif "IsHoliday_y" in df.columns:
            df["IsHoliday"] = df["IsHoliday_y"]
        else:
            raise KeyError("No IsHoliday column found.")

    drop_cols = [c for c in ["IsHoliday_x", "IsHoliday_y"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    df["IsHoliday"] = df["IsHoliday"].astype(bool)
    return df


def make_neuralforecast_df(
    df: pd.DataFrame,
    target_col: str = "Weekly_Sales_clipped",
    target_transform: str = "none",
) -> pd.DataFrame:
    """Convert Walmart dataframe to NeuralForecast format: unique_id, ds, y."""
    df = df.copy()

    df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
    df["ds"] = pd.to_datetime(df["Date"])

    # Rows created by fill_grid have missing original Weekly_Sales.
    df["was_missing_week"] = df["Weekly_Sales"].isna().astype(int)

    df = fill_missing_targets_for_series(
        df,
        target_col=target_col,
        strategy=CONFIG["missing_target_strategy"],
    )

    if target_transform == "log1p":
        df["y"] = np.log1p(df[target_col])
    elif target_transform == "none":
        df["y"] = df[target_col]
    else:
        raise ValueError(f"Unknown target_transform: {target_transform}")

    return df[["unique_id", "ds", "y", "IsHoliday"]].copy()


def inverse_target_transform(preds: pd.DataFrame, model_col: str, target_transform: str) -> pd.DataFrame:
    """Convert model predictions back to original sales scale."""
    preds = preds.rename(columns={model_col: "pred_transformed"}).copy()

    if target_transform == "log1p":
        preds["pred"] = np.expm1(preds["pred_transformed"])
    elif target_transform == "none":
        preds["pred"] = preds["pred_transformed"]
    else:
        raise ValueError(f"Unknown target_transform: {target_transform}")

    preds["pred"] = preds["pred"].clip(lower=0)
    return preds


def evaluate_wmae(eval_df: pd.DataFrame) -> Dict[str, float]:
    """Compute competition WMAE plus holiday/non-holiday breakdown."""
    y_true = eval_df["Weekly_Sales_clipped"].fillna(0).values
    y_pred = eval_df["pred"].fillna(0).values
    is_holiday = eval_df["IsHoliday"].values

    metrics = {
        "wmae": float(weighted_mae(y_true, y_pred, is_holiday)),
    }

    holiday_mask = eval_df["IsHoliday"] == True
    nonholiday_mask = eval_df["IsHoliday"] == False

    metrics["holiday_wmae"] = float(weighted_mae(
        eval_df.loc[holiday_mask, "Weekly_Sales_clipped"].fillna(0).values,
        eval_df.loc[holiday_mask, "pred"].fillna(0).values,
        eval_df.loc[holiday_mask, "IsHoliday"].values,
    ))

    metrics["nonholiday_wmae"] = float(weighted_mae(
        eval_df.loc[nonholiday_mask, "Weekly_Sales_clipped"].fillna(0).values,
        eval_df.loc[nonholiday_mask, "pred"].fillna(0).values,
        eval_df.loc[nonholiday_mask, "IsHoliday"].values,
    ))

    return metrics

def fill_missing_targets_for_series(df, target_col, strategy):
    df = df.copy()
    df = df.sort_values(["Store", "Dept", "Date"])

    if strategy == "zero":
        df[target_col] = df[target_col].fillna(0)

    elif strategy == "linear_interpolation":
        df[target_col] = (
            df.groupby(["Store", "Dept"])[target_col]
            .transform(lambda s: s.interpolate(method="linear", limit_area="inside"))
        )
        df[target_col] = df[target_col].fillna(0)

    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    return df


for name in ["train_part", "valid_part", "train_full", "test_full"]:
    globals()[name] = fix_isholiday_columns(globals()[name])

print("IsHoliday columns fixed.")
print("Validation horizon:", CONFIG["horizon"])
print("Number of Store-Dept series:", train_part[["Store", "Dept"]].drop_duplicates().shape[0])

IsHoliday columns fixed.
Validation horizon: 14
Number of Store-Dept series: 3331


## 6. Baseline: Store-Dept median

In [35]:
def store_dept_median_baseline(train_df: pd.DataFrame, actual_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    baseline = (
        train_df
        .groupby(["Store", "Dept"])["Weekly_Sales_clipped"]
        .median()
        .reset_index()
        .rename(columns={"Weekly_Sales_clipped": "pred"})
    )

    eval_df = actual_df.merge(baseline, on=["Store", "Dept"], how="left")
    eval_df["pred"] = eval_df["pred"].fillna(train_df["Weekly_Sales_clipped"].median())
    metrics = evaluate_wmae(eval_df)
    return eval_df, metrics


baseline_valid_eval, baseline_valid_metrics = store_dept_median_baseline(train_part, valid_part)

print(f"Baseline valid WMAE: {baseline_valid_metrics['wmae']:.4f}")
print(f"Baseline holiday WMAE: {baseline_valid_metrics['holiday_wmae']:.4f}")
print(f"Baseline non-holiday WMAE: {baseline_valid_metrics['nonholiday_wmae']:.4f}")

Baseline valid WMAE: 2027.5849
Baseline holiday WMAE: 2270.6346
Baseline non-holiday WMAE: 1934.1042


## 7. Train/evaluate helper

This helper trains N-BEATS on one time period and evaluates on the next horizon.

We use it twice:
- **train-tail check**: train on earlier training data and evaluate on the last 14 weeks inside the training period.
- **validation**: train on all `train_part` and evaluate on `valid_part`.

The train-tail metric is not the exact training loss. It is a clean generalization check on a held-out tail inside the training period.


In [36]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

def build_nbeats(config: Dict) -> NeuralForecast:
    model_kwargs = dict(
        h=config["horizon"],
        input_size=config["input_size"],
        loss=MAE(),
        max_steps=config["max_steps"],
        learning_rate=config["learning_rate"],
        batch_size=config["batch_size"],
        random_seed=config["random_seed"],
    )

    if config.get("scaler_type") is not None:
        model_kwargs["scaler_type"] = config["scaler_type"]

    model = NBEATS(**model_kwargs)

    return NeuralForecast(
        models=[model],
        freq=config["freq"],
    )


def fit_predict_evaluate(
    train_df_raw: pd.DataFrame,
    actual_df_raw: pd.DataFrame,
    config: Dict,
    label: str,
):
    train_nf = make_neuralforecast_df(
        train_df_raw,
        target_transform=config["target_transform"],
    )
    print(train_nf.columns.tolist())
    nf = build_nbeats(config)
    nf.fit(df=train_nf[["unique_id", "ds", "y"]])

    preds = nf.predict()
    preds = inverse_target_transform(
        preds,
        model_col=config["model"],
        target_transform=config["target_transform"],
    )

    actual = actual_df_raw.copy()
    actual["unique_id"] = actual["Store"].astype(str) + "_" + actual["Dept"].astype(str)
    actual["ds"] = pd.to_datetime(actual["Date"])

    eval_df = actual.merge(
        preds[["unique_id", "ds", "pred", "pred_transformed"]],
        on=["unique_id", "ds"],
        how="left",
    )

    eval_df["pred"] = eval_df["pred"].fillna(0)
    metrics = evaluate_wmae(eval_df)

    print(f"\n{label} metrics")
    print(f"WMAE: {metrics['wmae']:.4f}")
    print(f"Holiday WMAE: {metrics['holiday_wmae']:.4f}")
    print(f"Non-holiday WMAE: {metrics['nonholiday_wmae']:.4f}")

    return nf, preds, eval_df, metrics

## 8. Train-tail check

In [57]:
all_train_dates = sorted(train_part["Date"].unique())
tail_dates = all_train_dates[-CONFIG["horizon"]:]

train_inner = train_part[~train_part["Date"].isin(tail_dates)].copy()
train_tail = train_part[train_part["Date"].isin(tail_dates)].copy()

print("train_inner date range:", train_inner["Date"].min(), "→", train_inner["Date"].max())
print("train_tail date range:", train_tail["Date"].min(), "→", train_tail["Date"].max())
print("train_tail weeks:", train_tail["Date"].nunique())

baseline_train_tail_eval, baseline_train_tail_metrics = store_dept_median_baseline(train_inner, train_tail)
print(f"Baseline train-tail WMAE: {baseline_train_tail_metrics['wmae']:.4f}")

nf_train_tail, train_tail_preds, train_tail_eval, train_tail_metrics = fit_predict_evaluate(
    train_inner,
    train_tail,
    CONFIG,
    label="N-BEATS train-tail",
)

train_inner date range: 2010-02-05 00:00:00 → 2012-04-13 00:00:00
train_tail date range: 2012-04-20 00:00:00 → 2012-07-20 00:00:00
train_tail weeks: 14
Baseline train-tail WMAE: 2217.1026


/content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/preprocessing.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True


['unique_id', 'ds', 'y', 'IsHoliday']


INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
3.4 K     Non-trainable params
2.6 M     Total params
10.476    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=2000` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]


N-BEATS train-tail metrics
WMAE: 1765.1400
Holiday WMAE: nan
Non-holiday WMAE: 1765.1400


/content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/preprocessing.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


## 9. Final validation training and evaluation

In [58]:
nf_valid, valid_preds, valid_eval, valid_metrics = fit_predict_evaluate(
    train_part,
    valid_part,
    CONFIG,
    label="N-BEATS validation",
)

improvement = baseline_valid_metrics["wmae"] - valid_metrics["wmae"]
improvement_pct = 100 * improvement / baseline_valid_metrics["wmae"]
generalization_gap = valid_metrics["wmae"] - train_tail_metrics["wmae"]

summary = pd.DataFrame([
    {
        "split": "train_tail",
        "baseline_wmae": baseline_train_tail_metrics["wmae"],
        "model_wmae": train_tail_metrics["wmae"],
        "holiday_wmae": train_tail_metrics["holiday_wmae"],
        "nonholiday_wmae": train_tail_metrics["nonholiday_wmae"],
    },
    {
        "split": "valid",
        "baseline_wmae": baseline_valid_metrics["wmae"],
        "model_wmae": valid_metrics["wmae"],
        "holiday_wmae": valid_metrics["holiday_wmae"],
        "nonholiday_wmae": valid_metrics["nonholiday_wmae"],
    },
])

display(summary)

print(f"\nValidation improvement over baseline: {improvement:.4f} ({improvement_pct:.2f}%)")
print(f"Generalization gap valid - train_tail: {generalization_gap:.4f}")

if generalization_gap > 300:
    print("⚠️ Validation is much worse than train-tail. Possible overfitting or distribution shift.")
elif valid_metrics["wmae"] > baseline_valid_metrics["wmae"]:
    print("❌ Model is worse than baseline. This setup is probably underperforming.")
else:
    print("✅ Model is better than baseline and train-tail/valid gap is not extreme.")

INFO:lightning_fabric.utilities.seed:Seed set to 42


['unique_id', 'ds', 'y', 'IsHoliday']


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
3.4 K     Non-trainable params
2.6 M     Total params
10.476    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=2000` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]


N-BEATS validation metrics
WMAE: 1177.3630
Holiday WMAE: 1231.9801
Non-holiday WMAE: 1156.3565


,split,baseline_wmae,model_wmae,holiday_wmae,nonholiday_wmae
0,train_tail,2217.102593,1765.140021,NaN,1765.140021
1,valid,2027.584890,1177.363018,1231.980078,1156.356456



Validation improvement over baseline: 850.2219 (41.93%)
Generalization gap valid - train_tail: -587.7770
✅ Model is better than baseline and train-tail/valid gap is not extreme.


## 10. MLflow / DagsHub setup

In [59]:
import dagshub
import mlflow

dagshub.init(
    repo_owner="izere23",
    repo_name="ML-Final-Walmart-Recruiting-Store-Sales-Forecasting",
    mlflow=True,
)

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("NBEATS_Training")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

MLflow tracking URI: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow


## 11. Save artifacts and log one clean run

In [60]:
import json
from pathlib import Path

EXPERIMENT_NAME = "NBEATS_Training"

ARTIFACT_DIR = Path("artifacts") / RUN_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

valid_preds.to_csv(ARTIFACT_DIR / "valid_predictions.csv", index=False)
valid_eval.to_csv(ARTIFACT_DIR / "valid_eval.csv", index=False)
train_tail_eval.to_csv(ARTIFACT_DIR / "train_tail_eval.csv", index=False)
summary.to_csv(ARTIFACT_DIR / "metrics_summary.csv", index=False)

feature_summary = {
    "used_by_nbeats_v1": ["unique_id", "ds", "y"],
    "used_for_evaluation": ["IsHoliday"],
    "created_by_common_preprocessing_but_not_used_by_nbeats_v1": [
        "Temperature", "Fuel_Price", "MarkDown1", "MarkDown2", "MarkDown3",
        "MarkDown4", "MarkDown5", "MarkDown1_exists", "MarkDown2_exists",
        "MarkDown3_exists", "MarkDown4_exists", "MarkDown5_exists",
        "CPI", "Unemployment", "Type", "Size", "Year", "Month",
        "WeekOfYear", "Week_sin", "Week_cos", "is_superbowl",
        "is_labor_day", "is_thanksgiving", "is_christmas",
    ],
}

with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=4)

with open(ARTIFACT_DIR / "feature_summary.json", "w") as f:
    json.dump(feature_summary, f, indent=4)

MODEL_DIR = ARTIFACT_DIR / "model"

if mlflow.active_run():
    mlflow.end_run()

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(CONFIG)

    mlflow.log_metric("baseline_train_tail_wmae", baseline_train_tail_metrics["wmae"])
    mlflow.log_metric("train_tail_wmae", train_tail_metrics["wmae"])
    mlflow.log_metric("train_tail_holiday_wmae", train_tail_metrics["holiday_wmae"])
    mlflow.log_metric("train_tail_nonholiday_wmae", train_tail_metrics["nonholiday_wmae"])

    mlflow.log_metric("baseline_store_dept_median_wmae", baseline_valid_metrics["wmae"])
    mlflow.log_metric("valid_wmae", valid_metrics["wmae"])
    mlflow.log_metric("holiday_wmae", valid_metrics["holiday_wmae"])
    mlflow.log_metric("nonholiday_wmae", valid_metrics["nonholiday_wmae"])
    mlflow.log_metric("improvement_over_baseline", improvement)
    mlflow.log_metric("improvement_over_baseline_pct", improvement_pct)
    mlflow.log_metric("generalization_gap_valid_minus_train_tail", generalization_gap)

    mlflow.log_artifacts(str(ARTIFACT_DIR), artifact_path="artifacts")

    nf_valid.save(path=str(MODEL_DIR), overwrite=True)
    mlflow.log_artifacts(str(MODEL_DIR), artifact_path="model")

print("✅ Metrics, parameters, artifacts, and model were logged successfully.")

🏃 View run NBEATS_none_target_input104_h14_steps2000 at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/1/runs/7c003d848c98404ea419c9813484eaa3
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/1
✅ Metrics, parameters, artifacts, and model were logged successfully.


## 12. README notes

Copy this into README/report after each important run:

- N-BEATS v1 is a global univariate model.
- It uses only `unique_id`, `ds`, and `y`.
- It does not use MarkDown, weather, CPI, unemployment, store type, or calendar covariates as model inputs.
- The competition metric is WMAE: holiday weeks receive 5x weight.
- We added a train-tail check to compare validation performance against a previous held-out period from the training range.
- If validation WMAE is much higher than train-tail WMAE, that suggests overfitting or distribution shift.
- If both train-tail and validation WMAE are high and worse than baseline, the model is underperforming.
